In [1]:
print("ok")

ok


In [59]:
from dotenv import load_dotenv
import os


In [1]:
%pwd

'c:\\Users\\user\\Desktop\\Medical-chatbot\\research'

In [14]:
import os 
os.chdir("../")

In [15]:
%pwd

'c:\\Users\\user\\Desktop\\Medical-chatbot'

In [26]:
import os
os.chdir(r"C:\Users\user\Desktop\Medical-chatbot\research")

In [11]:
import os 
os.chdir("/..")

In [16]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [18]:
extracted_data=load_pdf_file(data='data/')

In [19]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [20]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 582


In [35]:
pip install langchain-huggingface

In [21]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test
query_result = embeddings.embed_query("Hello world")
print("Length:", len(query_result))  # Should print 384

c:\Users\user\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


RuntimeError: Failed to import transformers.models.bert.modeling_bert because of the following error (look up to see its traceback):
cannot import name 'split_torch_state_dict_into_shards' from 'huggingface_hub' (c:\Users\user\anaconda3\envs\medibot\lib\site-packages\huggingface_hub\__init__.py)

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(len(embeddings.embed_query("Hello world")))  # Should print 384

c:\Users\user\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\user\anaconda3\envs\medibot\lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


384


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import os

In [22]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
MISTRAL_API_KEY=os.environ.get('MISTRAL_API_KEY')

In [23]:
print("Pinecone API Key:", PINECONE_API_KEY)

Pinecone API Key: pcsk_788jBx_RHFoG2w8XB4vVHzZazk9oQj3HiEgXExzaKCqkWnQCemtXyDQtg9gTuFF2H4sD46


In [8]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

# Delete old index
pc.delete_index(index_name)

# Recreate with 384 dimensions for HuggingFace
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

print("Index recreated successfully!")

Index recreated successfully!


In [ ]:
pcsk_788jBx_RHFoG2w8XB4vVHzZazk9oQj3HiEgXExzaKCqkWnQCemtXyDQtg9gTuFF2H4sD46

In [24]:
from langchain_pinecone import PineconeVectorStore
import os

os.environ["PINECONE_API_KEY"] = "pcsk_788jBx_RHFoG2w8XB4vVHzZazk9oQj3HiEgXExzaKCqkWnQCemtXyDQtg9gTuFF2H4sD46"

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

print("Done!")

Done!


In [25]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [26]:
docsearch

In [27]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [28]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='a6e0051e-35e3-4b5d-89e4-8818687006c9', metadata={'author': 'Osmo Otto Paivio Hanninen', 'creationdate': '2011-02-08T18:56:56+04:00', 'creator': 'Acrobat PDFMaker 7.0 for Word', 'moddate': '2022-04-20T09:57:08+04:00', 'page': 5.0, 'page_label': '6', 'producer': 'Acrobat Distiller 7.0 (Windows)', 'source': 'data\\MEDICAL AND HEALTH SCIENCES.pdf', 'subject': 'MEDICAL AND HEALTH SCIENCES', 'title': 'MEDICAL AND HEALTH SCIENCES', 'total_pages': 117.0}, page_content='MEDICAL AND HEALTH SCIENCES \n2.2.4. Neuronal Repair \n2.3. Proliferative Phase: Collagen and Proteoglycan Synthesis and Wound Contraction \n2.3.1. Collagen and Proteoglycan Synthesis \n2.3.2. Wound Contraction \n2.4. Late Phase: Remodeling \n3. Pathologic Responses to Wounding \n3.1. General Health and Stress \n3.2. Nutrition \n3.3. Pharmacologic Impediments \n3.4. Predisposing Diseases \n3.4.1. Diabetes Mellitus \n3.4.2. Pressure Sores \n3.4.3. Venous Stasis Ulcers \n3.5. Overhealing Wounds'),
 Document(id='9ff8b

In [29]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    mistral_api_key="MsDafT0pV5kvWdXHtTY26blVLE1340WW",
    temperature=0.4,
    max_tokens=500
)

In [30]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Test it
response = rag_chain.invoke({"input": "What is your question here?"})
print(response["answer"])

The provided context does not contain a specific question. It appears to be a list of topics or sections from a document, possibly an e-book or a structured guide. The sections cover various research methods, media topics, human rights considerations, poverty-related issues, and macroeconomic policies. If you have a specific question related to any of these topics, please ask, and I will do my best to provide a concise answer based on the given context.


In [31]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [32]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly and gigantism are both conditions caused by excess growth hormone (GH) but they differ in the timing of the excess GH secretion. Acromegaly occurs when excess GH is produced after the growth plates in the bones have closed, leading to thickening of bones and tissues. Gigantism, on the other hand, occurs when excess GH is produced before the growth plates have closed, resulting in excessive linear growth and very tall stature. Both conditions are typically caused by a benign tumor on the pituitary gland that secretes excess GH.


In [33]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

I don't know.
